In [ ]:
import pandas as _hex_pandas
import datetime as _hex_datetime
import json as _hex_json

In [ ]:
hex_scheduled = _hex_json.loads("false")

In [ ]:
hex_user_email = _hex_json.loads("\"example-user@example.com\"")

In [ ]:
hex_user_attributes = _hex_json.loads("{}")

In [ ]:
hex_run_context = _hex_json.loads("\"logic\"")

In [ ]:
hex_timezone = _hex_json.loads("\"UTC\"")

In [ ]:
hex_project_id = _hex_json.loads("\"01a0d4c5-3039-71f5-82ae-7437d59555fe\"")

In [ ]:
hex_project_name = _hex_json.loads("\"Trial Conversion Signals\"")

In [ ]:
hex_status = _hex_json.loads("\"\"")

In [ ]:
hex_categories = _hex_json.loads("[]")

In [ ]:
hex_color_palette = _hex_json.loads("[\"#4C78A8\",\"#F58518\",\"#E45756\",\"#72B7B2\",\"#54A24B\",\"#EECA3B\",\"#B279A2\",\"#FF9DA6\",\"#9D755D\",\"#BAB0AC\"]")

# 1. Project Scope

* **Question:** Which self-service trial engagement signals predict a conversion to a paid subscription?
* **Objective:** Deliver insights on synthetic trial conversion data that empower the customer team to drive product adoption.

# 2. Generating the Synthetic Dataset

Generation of a synthetic dataset, necessary to walk through a trial conversion analysis

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 2000

signup_channel = rng.choice(
    ["organic_search", "paid_search", "referral", "content", "partner"],
    size=n, p=[0.35, 0.25, 0.15, 0.15, 0.10]
)
seats_invited = rng.poisson(2, n)
datasets_connected = rng.poisson(1.2, n)
queries_run_wk1 = rng.negative_binomial(2, 0.08, n)
support_tickets = rng.poisson(0.6, n)
days_active = np.clip(
    rng.binomial(14, 0.25, n) + (queries_run_wk1 > 20) * rng.integers(0, 5, n) + datasets_connected,
    1, 14
)

channel_effect = pd.Series(signup_channel).map(
    {"organic_search": 0.0, "paid_search": -0.3, "referral": 0.6, "content": 0.1, "partner": 0.4}
).to_numpy()

logit = (
    -3.2
    + 0.35 * np.minimum(seats_invited, 6)
    + 0.55 * np.minimum(datasets_connected, 4)
    + 0.015 * np.minimum(queries_run_wk1, 80)
    + 0.12 * days_active
    + 0.15 * np.minimum(support_tickets, 2) - 0.35 * np.maximum(support_tickets - 2, 0)
    + channel_effect
    + rng.normal(0, 0.5, n)
)
converted = rng.binomial(1, 1 / (1 + np.exp(-logit)))

trials_df = pd.DataFrame({
    "account_id": [f"ACC-{i:05d}" for i in range(1, n + 1)],
    "signup_channel": signup_channel,
    "seats_invited": seats_invited,
    "datasets_connected": datasets_connected,
    "queries_run_wk1": queries_run_wk1,
    "support_tickets": support_tickets,
    "days_active": days_active,
    "converted": converted,
})
trials_df

,account_id,signup_channel,seats_invited,datasets_connected,queries_run_wk1,support_tickets,days_active,converted
0,ACC-00001,content,4,0,3,0,6,0
1,ACC-00002,paid_search,2,2,37,2,7,1
2,ACC-00003,content,1,0,104,2,6,0
3,ACC-00004,referral,0,0,32,0,10,1
4,ACC-00005,organic_search,1,0,43,0,6,0
...,...,...,...,...,...,...,...,...
1995,ACC-01996,content,3,2,14,1,5,1
1996,ACC-01997,organic_search,2,4,54,0,10,1
1997,ACC-01998,organic_search,3,0,36,0,3,0
1998,ACC-01999,content,0,0,19,2,3,0


# 3. Describing the Dataset
Dataset profile for each signal to get a sense of the distribution of the data

In [ ]:
numeric_cols = [
    "seats_invited",
    "datasets_connected",
    "queries_run_wk1",
    "support_tickets",
    "days_active"
]

# Describe the dataset for each signal to get a sense of the distribution of the data
trials_df[numeric_cols].describe()

,seats_invited,datasets_connected,queries_run_wk1,support_tickets,days_active
count,2000.000000,2000.000000,2000.000000,2000.00000,2000.000000
mean,1.933500,1.217000,23.337500,0.56650,5.720000
std,1.402876,1.118721,17.097683,0.73814,2.409461
min,0.000000,0.000000,0.000000,0.00000,1.000000
25%,1.000000,0.000000,11.000000,0.00000,4.000000
50%,2.000000,1.000000,19.000000,0.00000,5.000000
75%,3.000000,2.000000,32.000000,1.00000,7.000000
max,7.000000,6.000000,157.000000,4.00000,14.000000


# 4. Evaluating Conversion and Ranking Signals
* Calculation of the overall conversion rate across the synthetic dataset
* Calculation of correlation to use as a baseline (doesn't yet adjust for confounding signals)
* Calculation of a standardized logistic regression to evaluate the impact of each signal in isolation (holding other signals that might correlate with it constant)

In [ ]:
overall_conversion_rate = trials_df['converted'].mean()
overall_conversion_rate

np.float64(0.3535)

In [ ]:
trials_df[numeric_cols].corrwith(trials_df["converted"])

seats_invited         0.161089
datasets_connected    0.266124
queries_run_wk1       0.123439
support_tickets       0.021946
days_active           0.259194
dtype: float64

In [ ]:
# Install and import statsmodels module
!uv pip install statsmodels==0.15.0 -q
import statsmodels.formula.api as smf

# Copy trials_df as z_df and standardize signals
z_df = trials_df.copy()

z_df[numeric_cols] = (trials_df[numeric_cols] - trials_df[numeric_cols].mean()) / trials_df[numeric_cols].std()

# Fit the model
model = smf.logit(
    "converted ~ seats_invited + datasets_connected + queries_run_wk1 + support_tickets + days_active + C(signup_channel, Treatment('organic_search'))", 
    data=z_df
).fit()

# Extract coefficients and 95% confidence intervals into a DataFrame
summary_df = pd.DataFrame({
    "coef": model.params,
    "conf_lower": model.conf_int().iloc[:, 0],
    "conf_upper": model.conf_int().iloc[:, 1],
})

# Filter to numeric columns only (excluding Intercept and categorical channels)
signals_df = summary_df.loc[numeric_cols].copy()

# Add the odds ratio, percentage change in odds, and confidence interval ratio
signals_df["odds_ratio"] = np.exp(model.params)
signals_df["odds_ratio_pct_change"] = (signals_df["odds_ratio"] - 1) * 100
signals_df["conf_lower_ratio"] = np.exp(signals_df["conf_lower"])
signals_df["conf_upper_ratio"] = np.exp(signals_df["conf_upper"])

# Add the absolute value of each coefficient for sorting purposes
signals_df["abs_coef"] = signals_df["coef"].abs()

# Sort by absolute magnitude descending
ranked_signals = signals_df.sort_values("abs_coef", ascending=False)
ranked_signals


Optimization terminated successfully.
         Current function value: 0.571047
         Iterations 5


,coef,conf_lower,conf_upper,odds_ratio,odds_ratio_pct_change,conf_lower_ratio,conf_upper_ratio,abs_coef
datasets_connected,0.490333,0.373984,0.606683,1.632860,63.285990,1.453513,1.834336,0.490333
seats_invited,0.431196,0.330637,0.531754,1.539096,53.909646,1.391855,1.701914,0.431196
days_active,0.332659,0.212247,0.453070,1.394671,39.467105,1.236453,1.573134,0.332659
queries_run_wk1,0.212280,0.108315,0.316245,1.236494,23.649425,1.114399,1.371966,0.212280
support_tickets,0.057540,-0.041467,0.156548,1.059228,5.922798,0.959381,1.169467,0.057540


In [ ]:
print(model.summary())

                           Logit Regression Results                           
Dep. Variable:              converted   No. Observations:                 2000
Model:                          Logit   Df Residuals:                     1990
Method:                           MLE   Df Model:                            9
Date:                Mon, 28 Sep 2026   Pseudo R-squ.:                  0.1209
Time:                        15:28:57   Log-Likelihood:                -1142.1
converged:                       True   LL-Null:                       -1299.2
Covariance Type:            nonrobust   LLR p-value:                 2.581e-62
                                                                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------------------------------------
Intercept                                                        -0.7163      0.086     -8.362      0.000   

# 5. Digging Deeper into Top Signal Data

* Initial bucketing of top signal by quantiles (however, the data is heavily skewed toward 0 datasets connected during trials)
* Final bucketing of top signal by explicit bins to get meaningful insight

In [ ]:
#Define buckets for top signal using quantiles
tiered_qcut_df = trials_df.copy()
tiered_qcut_df['engagement_tier'] = pd.qcut(tiered_qcut_df['datasets_connected'], q=3, labels =["1-low", "2-mid", "3-high"])

In [ ]:
#Define buckets for top signal using explicit bins since data is skewed toward 0; tested defining 3 bins initially, but found 4 bins showed the distribution better since there are also quite a few accounts with 0 datasets connected.
tiered_explicit_df = trials_df.copy()
tiered_explicit_df['engagement_tier'] = pd.cut(tiered_explicit_df['datasets_connected'], bins = [-1, 0, 1, 2, np.inf], labels =["0", "1", "2", "3+"])

# 6. Aggregating Top-Ranked Signal by Signup Channel and Conversion Rate
* Calculation of conversion rate and count of accounts by signup channel and top-ranked signal bins
* Visualization of distribution of top-ranked signal bins by conversion rate and signup channel
* Visualization of distribution of signup channels by conversion rate and top-ranked signal bins

In [ ]:
# import jinja2
# raw_query = """
#     SELECT 
#         signup_channel, 
#         engagement_tier, 
#         AVG(converted) AS conversion_rate, 
#         COUNT(*) AS n_accounts 
#     FROM tiered_explicit_df
#     GROUP BY signup_channel, engagement_tier
#     ORDER BY signup_channel, engagement_tier
# """
# sql_query = jinja2.Template(raw_query).render(vars())

# 7. Analysis Summary
## Signal Ranking
After ranking 5 signals in a synthetic trial conversion dataset by their standardized logistic regression coefficients, controlling for the other signals, the order of the signals' association with conversion is, from greatest to least:
1. Number of datasets that the customer connected to during the trial,
2. Number of seats added to the trial account,
3. Number of days that the customer interacted with the product during the trial,
4. Number of queries that the customer ran during the first week of the trial, and
5. Number of support tickets that the customer filed during the trial.

**A Note on Support Ticket Volume:** It's worth noting that while support ticket volume appears to have a fairly neutral effect on conversion (p-value of 0.26, far above the threshold for statistical significance), this is likely masking to some degree a correlation between distinct volumes of support tickets filed and conversion:
* A non-zero but small number of tickets likely indicates customer engagement and would be expected to correlate with conversion, while 
* A large number of tickets likely indicates that the customer is struggling with the product and would be expected to have an inverse relationship with conversion. 

We should explore this further if the support team sees trends in support ticket volumes that would merit further inspection of this relationship.

## Exploring the Strongest Signal
Digging a bit deeper into the dataset-connection and conversion correlation:
* Most accounts connected between 0 and 2 datasets, and this pattern holds within every signup channel. 
* Accounts with 2 or more datasets connected see a noticeable increase in conversion rate. 
* Partner and referral channels with 0 datasets connected convert on par with or outperform paid_search conversions with 1-2 datasets connected, worth keeping an eye on in case it's a trend. 
* Dataset connections corresponds to roughly a 63% increase in the odds of conversion for every one standard-deviation increase in datasets connected.

## Recommended Next Step
Test for a causal relationship between dataset connection quantity and conversion. Are we just seeing that already-engaged users tend to connect more datasets, or could connecting more datasets also offer users more opportunity to engage and convert? 

Specifically, test whether a prompt to add a second data source connection early in the trial period would improve conversion causally.

Built with Hex's AI for data generation, chart scaffolding, and statistical/troubleshooting support; I drove the SQL logic, bucketing decisions, and final analysis.